<a href="https://colab.research.google.com/github/pavanajayan/CNN_Car-classification/blob/main/transfer_learning_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import time
import random
from tensorflow.keras.datasets import cifar10

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.applications import VGG16

#DATASET LOADING

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

print(X_train.shape)
print(y_train.shape)

(50000, 32, 32, 3)
(50000, 1)


In [ ]:
y_train = y_train.squeeze()
y_test = y_test.squeeze()

SEED = 42
val_size = 5000
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X_train))
val_idx, train_idx = perm[:val_size], perm[val_size:]

x_train, y_tr = X_train[train_idx], y_train[train_idx]
x_val, y_val = X_train[val_idx], y_train[val_idx]

print(f"Train: {len(x_train)} | Val: {len(x_val)} | Test: {len(X_test)}")

Train: 45000 | Val: 5000 | Test: 10000


# DATA PREPROCESSING

In [ ]:
IMG_SIZE = 224          # Target size expected by the pretrained backbone (ImageNet-scale)
RESIZE_MARGIN = 32       # Extra pixels added before cropping
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

# ImageNet per-channel normalization stats
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD  = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]


random_rotation_layer = tf.keras.layers.RandomRotation(15 / 360.0, seed=SEED)


def normalize(image):
    # Scale pixels to [0,1]
    image = tf.cast(image, tf.float32) / 255.0
    return (image - IMAGENET_MEAN) / IMAGENET_STD


def resize_only(image):
    # Used for validation/test — deterministic resize, no augmentation,
    return tf.image.resize(image, [IMG_SIZE, IMG_SIZE])


def train_augment(image):

    # Resize larger than the target size first
    image = tf.image.resize(image, [IMG_SIZE + RESIZE_MARGIN, IMG_SIZE + RESIZE_MARGIN])

    # Randomly crop back down to the model's expected input size
    image = tf.image.random_crop(image, size=[IMG_SIZE, IMG_SIZE, 3])

    # Randomly mirror left-right
    image = tf.image.random_flip_left_right(image)

    # Random Rotation
    image = random_rotation_layer(image[None, ...], training=True)[0]

    return image


def make_dataset(x, y, training, batch_size=BATCH_SIZE):

    ds = tf.data.Dataset.from_tensor_slices((x, y))

    if training:
        # Shuffle only the training set
        ds = ds.shuffle(buffer_size=min(len(x), 10000), seed=SEED)

    def _map(image, label):
        # Branch: augment during training
        image = train_augment(image) if training else resize_only(image)
        image = normalize(image)
        return image, label

    return (
        ds.map(_map, num_parallel_calls=AUTOTUNE)
          .batch(batch_size)
          .prefetch(AUTOTUNE)
    )


# ============================================================
# Build the three pipelines
# ============================================================
train_ds = make_dataset(x_train, y_tr, training=True)     # augmented
val_ds   = make_dataset(x_val, y_val, training=False)      # clean, for early-stopping/checkpointing
test_ds  = make_dataset(X_test, y_test, training=False)    # clean, held out for final evaluation

#VGG16

In [ ]:
L2_LAMBDA = 1e-4

def build_model(num_classes=10, hidden_dim=512, dropout_p=0.5, l2_lambda=L2_LAMBDA):
    base_model = VGG16(
        weights="imagenet", include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg",
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.Dense(hidden_dim, kernel_initializer=keras.initializers.HeNormal(seed=SEED),
                      kernel_regularizer=regularizers.l2(l2_lambda))(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(dropout_p)(x)
    outputs = layers.Dense(num_classes, activation="softmax",
                            kernel_initializer=keras.initializers.HeNormal(seed=SEED),
                            kernel_regularizer=regularizers.l2(l2_lambda))(x)

    return keras.Model(inputs, outputs, name="vgg16_transfer"), base_model

model, base_model = build_model()
model.summary()

Model: "vgg16_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 512)            │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,984,522 (57.16 MB)

 Trainable params: 268,810 (1.03 MB)

 Non-trainable params: 14,715,712 (56.14 MB)

In [ ]:
def compile_model(model, lr, weight_decay):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def make_callbacks(path, patience=5, start_from_epoch=20):
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True,
            start_from_epoch=start_from_epoch,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
    ]

# FEATURE EXTRACTION

In [ ]:
FE_EPOCHS = 20
FE_LR = 1e-3
FE_WEIGHT_DECAY = 1e-4

model = compile_model(model, lr=FE_LR, weight_decay=FE_WEIGHT_DECAY)

fe_callbacks = make_callbacks("best_feature_extraction.weights.h5",
                               patience=5, start_from_epoch=20)

fe_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FE_EPOCHS,
    callbacks=fe_callbacks,
    verbose=1,
)

Epoch 1/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 656ms/step - accuracy: 0.5808 - loss: 1.3256
Epoch 1: val_loss improved from None to 0.86778, saving model to best_feature_extraction.weights.h5

Epoch 1: finished saving model to best_feature_extraction.weights.h5
704/704 ━━━━━━━━━━━━━━━━━━━━ 515s 696ms/step - accuracy: 0.6492 - loss: 1.1112 - val_accuracy: 0.7304 - val_loss: 0.8678 - learning_rate: 0.0010
Epoch 2/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 649ms/step - accuracy: 0.7117 - loss: 0.9167
Epoch 2: val_loss improved from 0.86778 to 0.84744, saving model to best_feature_extraction.weights.h5

Epoch 2: finished saving model to best_feature_extraction.weights.h5
704/704 ━━━━━━━━━━━━━━━━━━━━ 484s 687ms/step - accuracy: 0.7145 - loss: 0.9054 - val_accuracy: 0.7288 - val_loss: 0.8474 - learning_rate: 0.0010
Epoch 3/20
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 649ms/step - accuracy: 0.7321 - loss: 0.8602
Epoch 3: val_loss did not improve from 0.84744
704/704 ━━━━━━━━━━━━━━━━━━━━ 483s 686ms/step - accuracy:

In [ ]:
h = fe_history.history

# ---- Loss plot ----
plt.figure(figsize=(7, 5))
plt.plot(h["loss"], label="Train Loss")
plt.plot(h["val_loss"], label="Val Loss")
plt.title("Feature Extraction — Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

# ---- Accuracy plot ----
plt.figure(figsize=(7, 5))
plt.plot(h["accuracy"], label="Train Acc")
plt.plot(h["val_accuracy"], label="Val Acc")
plt.title("Feature Extraction — Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

# FINE TUNING

In [ ]:
def unfreeze_last_n_layers(base_model, n_layers, keep_bn_frozen=True):
    base_model.trainable = True
    freeze_until = len(base_model.layers) - n_layers
    for i, layer in enumerate(base_model.layers):
        layer.trainable = (i >= freeze_until) and not (keep_bn_frozen and isinstance(layer, layers.BatchNormalization))
    return [l.name for l in base_model.layers if l.trainable]

unfrozen = unfreeze_last_n_layers(base_model, n_layers=4)
print(unfrozen)

model = compile_model(model, lr=1e-5, weight_decay=1e-4)
ft_history = model.fit(train_ds, validation_data=val_ds, epochs=15,
                        callbacks=make_callbacks("best_ft.weights.h5", start_from_epoch=0), verbose=1)

## Fine-tuning curves + compare against feature extraction

In [ ]:
plot_history(ft_history, title_prefix="Fine-Tuning — ")

fe_val_loss_best = min(fe_history.history["val_loss"])
fe_val_acc_best = max(fe_history.history["val_accuracy"])
ft_val_loss_best = min(ft_history.history["val_loss"])
ft_val_acc_best = max(ft_history.history["val_accuracy"])

print(f"{'Phase':<20}{'Val Loss':<12}{'Val Acc':<12}")
print(f"{'Feature Extraction':<20}{fe_val_loss_best:<12.4f}{fe_val_acc_best:<12.4f}")
print(f"{'Fine-Tuning':<20}{ft_val_loss_best:<12.4f}{ft_val_acc_best:<12.4f}")

# EVALUATION

In [ ]:
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test

test_acc = (y_true == y_pred).mean()
print(f"Test accuracy: {test_acc:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — Test Set")
plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

per_class_acc = cm.diagonal() / cm.sum(axis=1)
hardest = sorted(zip(CLASS_NAMES, per_class_acc), key=lambda x: x[1])[:3]
print("Hardest classes:", hardest)